# PlayerPulse Data Profile

Bu notebook, WoWAH ham TXT dosyalarini parse edip ilk staging Parquet ciktilarini uretmek icin kullanilir.

In [31]:
import csv
import math
import re
from pathlib import Path

import duckdb
import pandas as pd

print('DuckDB ve pandas hazir.')


DuckDB ve pandas hazir.


In [32]:
def first_existing_path(candidates, *, is_dir=False, min_size_bytes=0):
    for path in candidates:
        if not path.exists():
            continue
        if is_dir and not path.is_dir():
            continue
        if not is_dir and path.stat().st_size <= min_size_bytes:
            continue
        return path
    return None


raw_root_candidates = [
    Path('/Users/osmanorka/Downloads/wowah/WoWAH'),
    Path.cwd() / 'wowah/WoWAH',
    Path.cwd().parent / 'wowah/WoWAH',
    Path.cwd() / 'data/raw/wowah/WoWAH',
    Path.cwd().parent / 'data/raw/wowah/WoWAH',
]

raw_root = first_existing_path(raw_root_candidates, is_dir=True)
if raw_root is None:
    raise FileNotFoundError(
        'WoWAH klasoru bulunamadi. Denenen konumlar: '
        + ', '.join(str(path) for path in raw_root_candidates)
    )

sample_file_candidates = [
    raw_root / '2006_01_03/2006-01-01/00-03-56.txt',
    Path('/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-03-56.txt'),
]

sample_file = first_existing_path(sample_file_candidates, min_size_bytes=1000)
if sample_file is None:
    raise FileNotFoundError('Ornek WoWAH TXT dosyasi bulunamadi.')

processed_dir = Path.cwd() / 'data/processed'
processed_dir.mkdir(parents=True, exist_ok=True)

print('Raw root:', raw_root)
print('Ornek dosya:', sample_file)
print('Ornek boyut:', f'{sample_file.stat().st_size:,} byte')
print('Processed dir:', processed_dir)


Raw root: /Users/osmanorka/Downloads/wowah/WoWAH
Ornek dosya: /Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-03-56.txt
Ornek boyut: 35,698 byte
Processed dir: /Users/osmanorka/PlayerPulse-LiveOps/notebooks/data/processed


In [33]:
def parse_wowah_file(file_path, verbose=False):
    file_path = Path(file_path)
    text = file_path.read_text(encoding='utf-8', errors='ignore')

    storage_match = re.search(r'Persistent_Storage\s*=\s*\{(.*?)\n\}', text, flags=re.DOTALL)
    if storage_match is None:
        raise ValueError(f'Persistent_Storage blogu bulunamadi: {file_path}')

    records = re.findall(r'\[\d+\]\s*=\s*"([^"]*)"', storage_match.group(1))

    rows = []
    skipped = []

    for raw_record in records:
        parts = next(csv.reader([raw_record], skipinitialspace=True))
        if len(parts) != 11:
            skipped.append(raw_record)
            continue
        rows.append([part.strip() for part in parts])

    columns = [
        'realm_or_source_id',
        'observed_at',
        'player_or_group_id',
        'avatar_id',
        'guild_id',
        'level',
        'race',
        'character_class',
        'zone',
        'guild_member_flag',
        'unknown_flag',
    ]

    df = pd.DataFrame(rows, columns=columns)

    if not df.empty:
        numeric_columns = [
            'realm_or_source_id',
            'player_or_group_id',
            'avatar_id',
            'guild_id',
            'level',
            'unknown_flag',
        ]
        df = df.replace({'': pd.NA})
        for column in numeric_columns:
            df[column] = pd.to_numeric(df[column], errors='coerce').astype('Int64')

        df['observed_at'] = pd.to_datetime(
            df['observed_at'],
            format='%m/%d/%y %H:%M:%S',
            errors='coerce',
        )
        df['guild_member_flag'] = (
            df['guild_member_flag']
            .astype('string')
            .str.lower()
            .map({'yes': True, 'no': False})
            .astype('boolean')
        )

    if verbose:
        print('Dosya:', file_path)
        print('Yakalanan kayit:', len(records))
        print('Okunan satir:', len(df))
        print('Atlanan kayit:', len(skipped))
        if skipped:
            print('Ilk atlanan kayit:', skipped[0])

    return df


In [34]:
with sample_file.open('r', encoding='utf-8', errors='ignore') as f:
    preview_lines = [line.rstrip() for _, line in zip(range(12), f)]

preview_lines


['',
 'Persistent_Storage = {',
 '\t[1] = "0, 12/31/05 23:59:46, 1,0, , 5, Orc, Warrior, Durotar, no, 0",',
 '\t[2] = "0, 12/31/05 23:59:46, 1,1, , 9, Orc, Shaman, Durotar, yes, 0",',
 '\t[3] = "0, 12/31/05 23:59:52, 2,2, , 13, Orc, Shaman, Durotar, no, 0",',
 '\t[4] = "0, 12/31/05 23:59:52, 2,3,0, 14, Orc, Warrior, Durotar, no, 0",',
 '\t[5] = "0, 12/31/05 23:59:52, 2,4, , 14, Orc, Shaman, Durotar, yes, 0",',
 '\t[6] = "0, 12/31/05 23:59:52, 2,5, , 16, Orc, Hunter, The Barrens, yes, 0",',
 '\t[7] = "0, 12/31/05 23:59:52, 2,6, , 18, Orc, Warlock, The Barrens, no, 0",',
 '\t[8] = "0, 12/31/05 23:59:52, 2,7, , 17, Orc, Hunter, Silverpine Forest, yes, 0",',
 '\t[9] = "0, 12/31/05 23:59:57, 3,8,0, 26, Orc, Warrior, Stonetalon Mountains, yes, 0",',
 '\t[10] = "0, 12/31/05 23:59:57, 3,9,1, 27, Orc, Hunter, Stonetalon Mountains, no, 0",']

In [35]:
df_sample = parse_wowah_file(sample_file, verbose=True)
df_sample.head(10)


Dosya: /Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-03-56.txt
Yakalanan kayit: 407
Okunan satir: 407
Atlanan kayit: 0


,realm_or_source_id,observed_at,player_or_group_id,avatar_id,guild_id,level,race,character_class,zone,guild_member_flag,unknown_flag
0,0,2005-12-31 23:59:46,1,0,<NA>,5,Orc,Warrior,Durotar,False,0
1,0,2005-12-31 23:59:46,1,1,<NA>,9,Orc,Shaman,Durotar,True,0
2,0,2005-12-31 23:59:52,2,2,<NA>,13,Orc,Shaman,Durotar,False,0
3,0,2005-12-31 23:59:52,2,3,0,14,Orc,Warrior,Durotar,False,0
4,0,2005-12-31 23:59:52,2,4,<NA>,14,Orc,Shaman,Durotar,True,0
5,0,2005-12-31 23:59:52,2,5,<NA>,16,Orc,Hunter,The Barrens,True,0
6,0,2005-12-31 23:59:52,2,6,<NA>,18,Orc,Warlock,The Barrens,False,0
7,0,2005-12-31 23:59:52,2,7,<NA>,17,Orc,Hunter,Silverpine Forest,True,0
8,0,2005-12-31 23:59:57,3,8,0,26,Orc,Warrior,Stonetalon Mountains,True,0
9,0,2005-12-31 23:59:57,3,9,1,27,Orc,Hunter,Stonetalon Mountains,False,0


In [36]:
print('Satir sayisi:', len(df_sample))
print('Unique player_or_group_id:', df_sample['player_or_group_id'].nunique())
print('Unique avatar_id:', df_sample['avatar_id'].nunique())
print('Duplicate observed_at + avatar_id:', df_sample.duplicated(subset=['observed_at', 'avatar_id']).sum())
print('Tarih min:', df_sample['observed_at'].min())
print('Tarih max:', df_sample['observed_at'].max())
print('Level min:', df_sample['level'].min())
print('Level max:', df_sample['level'].max())

df_sample.info()


Satir sayisi: 407
Unique player_or_group_id: 44
Unique avatar_id: 407
Duplicate observed_at + avatar_id: 0
Tarih min: 2005-12-31 23:59:46
Tarih max: 2006-01-01 00:03:43
Level min: 1
Level max: 60
<class 'pandas.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   realm_or_source_id  407 non-null    Int64         
 1   observed_at         407 non-null    datetime64[us]
 2   player_or_group_id  407 non-null    Int64         
 3   avatar_id           407 non-null    Int64         
 4   guild_id            338 non-null    Int64         
 5   level               407 non-null    Int64         
 6   race                407 non-null    str           
 7   character_class     407 non-null    str           
 8   zone                407 non-null    str           
 9   guild_member_flag   407 non-null    boolean       
 10  unknown_flag        407 non-null 

In [37]:
df_sample[['race', 'character_class', 'zone', 'guild_member_flag']].head(20)


,race,character_class,zone,guild_member_flag
0,Orc,Warrior,Durotar,False
1,Orc,Shaman,Durotar,True
2,Orc,Shaman,Durotar,False
3,Orc,Warrior,Durotar,False
4,Orc,Shaman,Durotar,True
5,Orc,Hunter,The Barrens,True
6,Orc,Warlock,The Barrens,False
7,Orc,Hunter,Silverpine Forest,True
8,Orc,Warrior,Stonetalon Mountains,True
9,Orc,Hunter,Stonetalon Mountains,False


In [38]:
df_sample.duplicated(subset=['observed_at', 'avatar_id']).sum()


np.int64(0)

In [39]:
txt_files = sorted(raw_root.rglob('*.txt'))
print('Toplam txt dosyasi:', len(txt_files))
for file in txt_files[:10]:
    print(file)


Toplam txt dosyasi: 138084
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-03-56.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-13-43.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-23-48.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-33-48.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-43-42.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-53-45.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/01-03-43.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/01-13-47.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/01-23-45.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/01-33-41.txt


In [40]:
sample_files = txt_files[:10]
dfs = []

for file in sample_files:
    df = parse_wowah_file(file, verbose=False)
    if df.empty:
        continue
    df['source_file'] = str(file.relative_to(raw_root))
    dfs.append(df)

df_test = pd.concat(dfs, ignore_index=True)
print('Toplam satir:', len(df_test))
print('Unique avatar:', df_test['avatar_id'].nunique())
print('Tarih min:', df_test['observed_at'].min())
print('Tarih max:', df_test['observed_at'].max())
df_test.head()


Toplam satir: 4057
Unique avatar: 597
Tarih min: 2005-12-31 23:59:46
Tarih max: 2006-01-01 01:33:28


,realm_or_source_id,observed_at,player_or_group_id,avatar_id,guild_id,level,race,character_class,zone,guild_member_flag,unknown_flag,source_file
0,0,2005-12-31 23:59:46,1,0,<NA>,5,Orc,Warrior,Durotar,False,0,2006_01_03/2006-01-01/00-03-56.txt
1,0,2005-12-31 23:59:46,1,1,<NA>,9,Orc,Shaman,Durotar,True,0,2006_01_03/2006-01-01/00-03-56.txt
2,0,2005-12-31 23:59:52,2,2,<NA>,13,Orc,Shaman,Durotar,False,0,2006_01_03/2006-01-01/00-03-56.txt
3,0,2005-12-31 23:59:52,2,3,0,14,Orc,Warrior,Durotar,False,0,2006_01_03/2006-01-01/00-03-56.txt
4,0,2005-12-31 23:59:52,2,4,<NA>,14,Orc,Shaman,Durotar,True,0,2006_01_03/2006-01-01/00-03-56.txt


In [41]:
print('Duplicate observed_at + avatar_id:', df_test.duplicated(subset=['observed_at', 'avatar_id']).sum())
print('Level range:', df_test['level'].min(), df_test['level'].max())
print('Guild missing rate:', df_test['guild_id'].isna().mean())
print('\nRace counts:')
print(df_test['race'].value_counts().head(10))
print('\nClass counts:')
print(df_test['character_class'].value_counts().head(10))


Duplicate observed_at + avatar_id: 0
Level range: 1 60
Guild missing rate: 0.18387971407443923

Race counts:
race
Undead    1409
Tauren    1140
Troll      806
Orc        689
373族        10
547人         3
Name: count, dtype: int64

Class counts:
character_class
Warrior    748
Hunter     687
Shaman     532
Rogue      509
Mage       501
Priest     453
Warlock    325
Druid      301
482          1
Name: count, dtype: int64


In [42]:
valid_races = {'Orc', 'Tauren', 'Troll', 'Undead'}
valid_classes = {'Warrior', 'Hunter', 'Shaman', 'Rogue', 'Mage', 'Priest', 'Warlock', 'Druid'}

df_test_anomalies = df_test[
    ~df_test['race'].isin(valid_races)
    | ~df_test['character_class'].isin(valid_classes)
]

print('Anomali satir sayisi:', len(df_test_anomalies))
df_test_anomalies[
    ['observed_at', 'player_or_group_id', 'avatar_id', 'race', 'character_class', 'zone', 'source_file']
].head(20)


Anomali satir sayisi: 14


,observed_at,player_or_group_id,avatar_id,race,character_class,zone,source_file
373,2006-01-01 00:03:22,41,373,373族,Mage,Badlands,2006_01_03/2006-01-01/00-03-56.txt
774,2006-01-01 00:13:09,41,373,373族,Mage,Badlands,2006_01_03/2006-01-01/00-13-43.txt
1179,2006-01-01 00:23:14,41,373,373族,Mage,Badlands,2006_01_03/2006-01-01/00-23-48.txt
1578,2006-01-01 00:33:15,41,373,373族,Mage,Badlands,2006_01_03/2006-01-01/00-33-48.txt
1814,2006-01-01 00:41:39,24,482,Troll,482,Orgrimmar,2006_01_03/2006-01-01/00-43-42.txt
1981,2006-01-01 00:43:09,41,373,373族,Mage,Badlands,2006_01_03/2006-01-01/00-43-42.txt
2380,2006-01-01 00:53:12,41,373,373族,Mage,Tirisfal Glades,2006_01_03/2006-01-01/00-53-45.txt
2792,2006-01-01 01:03:10,41,373,373族,Mage,Scarlet Monastery,2006_01_03/2006-01-01/01-03-43.txt
2994,2006-01-01 01:11:15,19,547,547人,Hunter,Durotar,2006_01_03/2006-01-01/01-13-47.txt
3204,2006-01-01 01:13:13,41,373,373族,Mage,Scarlet Monastery,2006_01_03/2006-01-01/01-13-47.txt


In [43]:
batch_size = 500
total_files = len(txt_files)
num_batches = math.ceil(total_files / batch_size)

print('Toplam dosya:', total_files)
print('Batch size:', batch_size)
print('Batch sayisi:', num_batches)
print('Cikti klasoru:', processed_dir)


Toplam dosya: 138084
Batch size: 500
Batch sayisi: 277
Cikti klasoru: /Users/osmanorka/PlayerPulse-LiveOps/notebooks/data/processed


In [44]:
def export_wowah_batches(files, output_dir, batch_size=500, start_batch=0, max_batches=None):
    def write_parquet_with_duckdb(df, output_path, view_name):
        con = duckdb.connect()
        try:
            con.register(view_name, df)
            con.execute(f"COPY {view_name} TO '{output_path}' (FORMAT PARQUET)")
        finally:
            con.close()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    total_files = len(files)
    num_batches = math.ceil(total_files / batch_size)
    stop_batch = num_batches if max_batches is None else min(num_batches, start_batch + max_batches)

    exported = []
    failures = []

    for batch_idx in range(start_batch, stop_batch):
        start = batch_idx * batch_size
        end = min(start + batch_size, total_files)
        batch_files = files[start:end]
        batch_dfs = []

        print(f'Batch {batch_idx + 1}/{num_batches} isleniyor. Dosya {start} - {end - 1}')

        for file in batch_files:
            try:
                df = parse_wowah_file(file, verbose=False)
                if df.empty:
                    continue
                df['source_file'] = str(file.relative_to(raw_root))
                batch_dfs.append(df)
            except Exception as exc:
                failures.append({'file': str(file), 'error': str(exc)})

        if not batch_dfs:
            print('Bu batch icin yazilacak veri yok.')
            continue

        batch_df = pd.concat(batch_dfs, ignore_index=True)
        output_path = output_dir / f'stg_wowah_events_part_{batch_idx:04d}.parquet'
        write_parquet_with_duckdb(batch_df, output_path, 'batch_df_view')

        exported.append({
            'batch_idx': batch_idx,
            'output_path': str(output_path),
            'row_count': len(batch_df),
            'file_count': len(batch_files),
        })
        print(f'Kaydedildi: {output_path} | Satir: {len(batch_df):,}')

    return pd.DataFrame(exported), pd.DataFrame(failures)


In [45]:
# Hazir oldugunda tum dataset icin calistir:
# export_summary, export_failures = export_wowah_batches(
#     files=txt_files,
#     output_dir=processed_dir,
#     batch_size=500,
# )
# export_summary.head()
